# Week 3  : Hill-Climbing and Simulated Annealing

In this tutorial you will practice writing local-search algorithms to solve a travelling-salesman problem in Python.

If you are new to Python don't worry. The tutorial will first walk you through an example of creating  hill-climber. Even if you know Python, you still should go over these steps.  The  first part of the tutorial covers

1. How to create random instances of TSP problems, and visualise them
2. Trying to solve small instances using an exact method
3. Writing a hill-climber to solver TSP (step by step instructions)
4. Some plotting functions fo visualise what's happening

Once you have completed this part, there are some tasks you need to complete (all described in the notebook below):

1. Conduct some experiments to understand the variance in results produced
2. Write your own **neighbourhood operator** and repeat the hill-climbing experiments
3. Write a **simulated-annealing algorithm**
4. Conduct some experiments to compare simulated annealing to a hill-climber on the same problem

Don't forget to "run" each cell in order to execute the code


## Setting things up

The main goal of the tutorial is to understand how to code local search algorithms so I've provided some code to manage the "extra' things that are required such as generating TSP instances and visualising them.  

For this lab, you can just **run** the cell below and not worry about the code itself. However if you are interested, I've provided an additional notebook that explains the code step by step (available in the Learning Space)

In [ ]:
# first import some libraries we will need

import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cmx
import operator, time, itertools, math
from random import randint


City = complex # Constructor for new cities, e.g. City(300, 400)

def distance(A, B): 
    "The Euclidean distance between two cities."
    return abs(A - B)

def generate_cities(n):
    "Make a set of n cities, each with random coordinates."
    return set(City(random.randrange(10, 890), 
                    random.randrange(10, 590)) 
                for c in range(n))

def getXY(points):
    "Given a list of points, return two lists: X coordinates, and Y coordinates."
    return [p.real for p in points], [p.imag for p in points]

def make_tour(cities):
    tour = list()
    for city in cities:
        tour.append(city)
    return tour

# helper function for plotting - this just draws a line between two points
    
def plotline(points, style='bo-', alpha=1, size=7, color=None):
    "Plot a list of points (complex numbers) in the 2-D plane."
    X, Y = getXY(points)
    
    if color:
        plt.plot(X, Y, style, alpha=alpha, markersize=size, color=color)
    else:
        plt.plot(X, Y, style, alpha=alpha, markersize=size)
        
        
def plot_tour(tour, alpha=1, color=None):
    # Plot the tour (starting city is shown as a square)
    plotline(list(tour) + [tour[0]], alpha=alpha, color=color)
    plotline([tour[0]], style='gD', alpha=alpha, size=10)
    # plt.show()
    
    
def all_non_redundant_tours(cities):
    "Return a list of tours, each a permutation of cities, but each one starting with the same city."
    start = first(cities)
    return [[start] + list(tour)
            for tour in itertools.permutations(cities - {start})]

def first(collection):
    "Start iterating over collection, and return the first element."
    for x in collection: return x

def exact_non_redundant_TSP(cities):
    "Generate all possible tours of the cities and choose the shortest one."
    return shortest(all_non_redundant_tours(cities))

def total_distance(tour):
    "The total distance between each pair of consecutive cities in the tour."
    return sum(distance(tour[i], tour[i-1]) 
               for i in range(len(tour)))

def shortest(tours): 
    "Return the tour with the minimum total distance."
    return min(tours, key=total_distance)


def make_random_tour(cities):
    tour = make_tour(cities)
    random_t = np.random.permutation(tour)
    return random_t

## Create a test instance
The *generate_cities(n)* method generates *n* cities with random coordinates.
The *getXY()* method provides the coordinates of the cities in two lists, x and y

Run the cell a few times to observe that it generates random coordinates

In [ ]:
cities10 = generate_cities(10)
x,y=getXY(cities10)
x,y

##  View the test instance
We can have a look at the 2d map of the cities we have just created using Python's plot function

In [ ]:
X, Y = getXY(cities10)
    
plt.plot(X, Y, 's', color='black')

## Step 2: Creating a random tour
Now we have the cities, we can make a tour around them.

The *make_random_tour()* function creates a random permutation of the cities you just generated.

The *total_distance()* function calculates the total distance of a given permutation

The *plot_tour()* function plots the random tour

If you can run the two cells below several times - and see examples of the different tours generated when it is passed a set of cities.

Again, run the cell a few times to verify

In [ ]:
random_t =  make_random_tour(cities10)
plot_tour(random_t)

total_distance(random_t)


## Generating all possible tours (small problems only!!)


How many possible tours are there? 

We can generate **all possible** permutations of a set of cities.    But recall in the lecture we discussed that this type of problem suffers from a combinatorial explosion so most problems are too difficult to solve like this.

You might also remember that actually, not all of these tours are unique:  the tours 1-2-3-4-5,  2-3-4-5-1, 3-4-5-1-2, 4,5,1,2,3 and 5,1,2,3,4 actually represent exactly the same sequence.  If we want to use an **exact** method to solve the problem (one that examines every possible solution) we can save ourselves a lot of time by removing all the redundant tours.


The function **exact_non_redundant_TSP()** generates all the non-redundant tours and then finds the shortest one

 
    

## How long does it take?

Now we have the tools, let's generate some cities, generate all non redundant tours and find the shortest. We will plot the result and also time to to see how long it takes:




1.  Run the code below for 5 cities
2.  Repeat for 8 cities and 10 cities

<font color=red> Warning... don't try with more cities than this!!  (in the table I have filled in the time obtained for 12 cities on my computer (MacBook Pro) for 12 cities and extrapolated this to 24) </font>


| n cities | average time |
| ---- | ---- |
| 5  | |
| 8  | |
| 10 |  |
| 12 | 2 mins and 1 second |
| 14 | ~20 hours |
| 24 | ~16 billion years |



**<font color=red>Clearly this is not practical!! Instead you now going to develop some local search methods as explained in the lecture</font>**!


In [ ]:
cities = generate_cities(5)
best = exact_non_redundant_TSP(cities)
plot_tour(best)

# the timing loop repeats the function 10 times so we can get an average
%timeit -n 1 -r 10 exact_non_redundant_TSP(cities)

##  Writing a hill-climber to solve TSP

We now have nearly all the tools need to write a hill-climber:
- a function to generate x cities
- a function to generate a random permutation of a tour
- a function to calculate the total distance of the tour

We just need a **neighbourhood operator** to make a change to an existing permutation.

The first operator we will try is the **swap function** as discussed in the lecture and given below. The function is passed an existing tour and then returns a modified version.  Notice that it first copies the existing tour and modifies the copy - this is important, as we don't want to over-write the existing tour


In [ ]:
def apply_neighbourhood_swap(individual):
    new_tour = individual.copy()
    position1 = randint(0,len(individual)-1)
    position2 = randint(0,len(individual)-1)
    city1 = individual[position1]
    city2 = individual[position2]
    new_tour[position1] = city2
    new_tour[position2] = city1
    
    return new_tour

Now we can define the hill-climber  function (refer back to the lecture notes if you are unsure)

The function is passed a random start tour, and iterates for a fixed number of trials, returning the best tour found.

A reminder of the algorithm:

+ set a random start tour (by randomising the tour above using np.random.permutation(start_tour) 
+ calculate distance of start tour
+ loop until max iterations:
    - copy tour then apply a **neighbourhood operator** to change it
    - calculate distance of modified tour
    - if modified tour is better than existing tour, replace the existing tour with the new one, and update the best fitness




In [ ]:
def hill_climber(tour):


    # and note the distance for this 
    best_tour = tour
    best_dist = total_distance(tour)


    # set the maximum number of tries
    max_tries = 10000


    # create an list and add the fitness of the starting tour (this is just for the purpose of plotting later)
    fitness_vals = list()
    fitness_vals.append(best_dist)
    initial_length = best_dist
    
    # now run the HC loop : 
    for i in range(max_tries-1):
        new_tour = apply_neighbourhood_swap(best_tour)
        new_tour_dist = total_distance(new_tour)
        if (new_tour_dist < best_dist):
            best_dist = new_tour_dist
            best_tour= new_tour.copy()
        
        #  add the current fitness to the list   (so we can plot it later)
        fitness_vals.append(best_dist)
        
    print("Starting tour length %s Best found %s  improvement of %s" % (initial_length, best_dist, initial_length-best_dist))  
   
    # return the results
    return best_dist, best_tour, fitness_vals



## Let's run it!
Now we are ready to create a test instance and then run the hill-climber:

In [ ]:
# CODE TO RUN THE HILL CLIMBER

# create a 50 tour problem 
test_instance = generate_cities(50)

# start from a random version of this tour
start_tour = make_random_tour(test_instance)

In [ ]:
# pass this randomised tour to the hill-climber

results =[]
for i in range(0,3):
    best_dist, best_tour, fits = hill_climber(start_tour)
    results.append(best_dist)
    
np.mean(results)

You can plot the best tour found and print the best distance.  
**Notice that if you run the cell above that calls the hill climber multiple times (starting from the same initial start_tour) you will get a different answer each time**

In [ ]:
plot_tour(best_tour)
print("shortest distance found is " , best_dist)


.. and also plot the change in fitness overtime

In [ ]:
plt.plot(fits)

## Task 1:   how much variation is there when the hill-climber is run multiple times?

1.  As the hill-climber is stochastic, it can produce a different result every time it is run. In the cell below, first copy the code above that calls the hill-climber and then modify it so that instead of running it once, it calls it 10 times.  On each run, start from a different randomised version of the start-tour.
Each time record the best fitness found and add it to a list. At the end, calculate the mean and standard deviation of the 10 runs. What do you notice? What is the best distance? What is the worst distance?

2.  Repeat this exercise with a problem that has 100 cities. What do you notice about the standard deviation in results?

3. Write a new neighbourhood operator of you choosing. It might invert a random segment of the tour, or apply multiple swaps, or.....     If you are feeling confident, then you can implement the **2-opt operator**. This is the best-known operator for TSP -  see https://en.wikipedia.org/wiki/2-opt for a description



In [ ]:
# Task 1 (a) run hill-climber multiple times and calculate statistics over runs

In [ ]:
# Task 1 (b) repeat using 100 cities

In [ ]:
# Task 1 (c) write a new neighbourhood operator

## Task 2:  Implementing a Simulated Annealing Algorithm

In the lecture we discussed the Simulated Annealing which extends the hill-climber by accepting solutions that are worse than the current solution, based on a calculation that includes a temperature factor and the difference in fitness between the current and potential solutions.

**Now implement an Simulated Annealing algorithm** (make a copy of the hill-climber code and start from here). Refer to the lecture slides for the algorithm

You will need to:
 - add a *temperature* variable, that can be initialised to a value of your choice (*start_temp*)
 - reduce the temperature by a small amount each step (the *cooling_rate*): for example, you can multiple the temperature variable by (say) 0.95 at each step which reduces it by 5%, or you can subtract a fixed amount from it. Generally reducing it slowly works best
 - calculate the probability of acceptance using the temperature and the difference in fitness (see the lecture slide with the equation)
 
 To determine whether to accept a new solution:
  - generate a random number between 0 and 1 using a python random number generator
  - if this random number is less than the probability calculated in the previous step, accept the solution

Test your algorithm on a 50 city problem. Suggested values for algorithm parameters are:
    - start temperature:  
    - cooling rate:
    
**Experimenting with your algorithm**
 - experiment with altering the starting temperature: what happens if it is very high? very low?
 - experiment with the cooling rate - what happens if you cool very quickly?


In [ ]:
# Task 2

#def sim_anneal(tour):


## Task 3: Comparing a hill-climber to simulated annealing

When you have found good parameters for the simulated annealing algorithm, compare the average fitness from 10 runs of the SA algorithm to 10 runs of the Hillclimber. Which one finds a better result?